# 02 — Groundwater drought events and metrics

This notebook derives well-level and aquifer-group drought events and metrics from the revised SGI series.

Definitions:
- drought month: SGI < -1.0;
- event: consecutive drought months; a missing month terminates an event;
- CRT = 1 - drought_months / valid_months;
- CRS = drought-to-nondrought transitions divided by drought months with a consecutive next observation;
- recovery: months from the event minimum to the first later consecutive month with SGI >= -0.5;
- propagation: an SGI-3 event is matched to an SGI-12 event when they overlap or when SGI-12 begins 0-12 months after SGI-3 onset.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

ROOT=Path.cwd()
if not (ROOT/"data_processed").exists() and (ROOT.parent/"data_processed").exists():
    ROOT=ROOT.parent
PROCESSED=ROOT/"data_processed"
INPUT=ROOT/"data_input"

sgi=pd.read_csv(PROCESSED/"SGI_normal_score_long.csv", parse_dates=["date"], float_precision="round_trip")
group_sgi=pd.read_csv(PROCESSED/"group_SGI.csv", parse_dates=["date"], float_precision="round_trip")
metadata=pd.read_csv(INPUT/"metadata_piezometers.csv", float_precision="round_trip")

print(f"Well-level SGI rows: {len(sgi):,}")
print(f"Group-level SGI rows: {len(group_sgi):,}")

In [ ]:
def month_diff(start, end):
    return (end.year-start.year)*12 + (end.month-start.month)

def detect_events(df, value_col, id_fields):
    df=df.sort_values("date").copy()
    dates=df["date"].tolist()
    vals=df[value_col].to_numpy(float)
    rows=[]
    event_id=0
    i=0
    while i<len(df):
        if vals[i] < -1.0:
            start_i=i
            j=i
            while j+1<len(df) and month_diff(dates[j],dates[j+1])==1 and vals[j+1] < -1.0:
                j+=1
            event_id+=1
            ev_vals=vals[start_i:j+1]
            min_i=start_i+int(np.argmin(ev_vals))
            recovery_date=pd.NaT
            recovery_months=np.nan
            recovered=False
            k=min_i+1
            previous=dates[min_i]
            while k<len(df):
                if month_diff(previous,dates[k]) != 1:
                    break
                if vals[k] >= -0.5:
                    recovery_date=dates[k]
                    recovery_months=month_diff(dates[min_i],dates[k])
                    recovered=True
                    break
                previous=dates[k]
                k+=1
            rows.append({
                **id_fields,
                "event_id":event_id,
                "start":dates[start_i],
                "end":dates[j],
                "duration_months":j-start_i+1,
                "minimum_date":dates[min_i],
                "minimum_sgi":float(vals[min_i]),
                "severity_signed":float(ev_vals.sum()),
                "severity_deficit":float(-ev_vals.sum()),
                "mean_intensity":float(ev_vals.mean()),
                "recovery_date":recovery_date,
                "recovery_months":recovery_months,
                "recovered":recovered,
            })
            i=j+1
        else:
            i+=1
    return rows

def compute_metrics(series_df, events_df, value_col, id_fields):
    x=series_df.sort_values("date").copy()
    dates=x["date"].tolist()
    vals=x[value_col].to_numpy(float)
    valid_months=len(x)
    drought_months=int((vals < -1.0).sum())
    eligible=transitions=0
    for i in range(len(x)-1):
        if vals[i] < -1.0 and month_diff(dates[i],dates[i+1])==1:
            eligible+=1
            if vals[i+1] >= -1.0:
                transitions+=1
    if len(events_df):
        recovered_events=int(events_df["recovered"].sum())
        rec={
            "N_events":len(events_df),
            "mean_duration":events_df["duration_months"].mean(),
            "max_duration":events_df["duration_months"].max(),
            "mean_severity_signed":events_df["severity_signed"].mean(),
            "mean_severity_deficit":events_df["severity_deficit"].mean(),
            "mean_intensity":events_df["mean_intensity"].mean(),
            "minimum_sgi":events_df["minimum_sgi"].min(),
            "VUL":events_df["severity_deficit"].max(),
            "median_recovery_months":events_df.loc[events_df["recovered"],"recovery_months"].median() if recovered_events else np.nan,
            "recovered_events":recovered_events,
            "unrecovered_events":len(events_df)-recovered_events,
        }
    else:
        rec={"N_events":0,"mean_duration":np.nan,"max_duration":np.nan,
             "mean_severity_signed":np.nan,"mean_severity_deficit":np.nan,
             "mean_intensity":np.nan,"minimum_sgi":np.nan,"VUL":np.nan,
             "median_recovery_months":np.nan,"recovered_events":0,"unrecovered_events":0}
    return {
        **id_fields,
        "valid_months":valid_months,
        "drought_months":drought_months,
        "drought_fraction":drought_months/valid_months if valid_months else np.nan,
        **{k:rec[k] for k in ["N_events","mean_duration","max_duration","mean_severity_signed","mean_severity_deficit","mean_intensity","minimum_sgi"]},
        "CRT":1-drought_months/valid_months if valid_months else np.nan,
        "CRS":transitions/eligible if eligible else np.nan,
        "VUL":rec["VUL"],
        "median_recovery_months":rec["median_recovery_months"],
        "recovered_events":rec["recovered_events"],
        "unrecovered_events":rec["unrecovered_events"],
    }

def propagation_matches(events, entity_cols):
    rows=[]
    for keys,evs in events.groupby(entity_cols,sort=True):
        if not isinstance(keys,tuple):
            keys=(keys,)
        ids=dict(zip(entity_cols,keys))
        e3=evs[evs["window_months"]==3].sort_values("start")
        e12=evs[evs["window_months"]==12].sort_values("start")
        for _,a in e3.iterrows():
            candidates=[]
            for _,b in e12.iterrows():
                overlap=(b["start"]<=a["end"]) and (b["end"]>=a["start"])
                lag=month_diff(a["start"],b["start"])
                if overlap or (0<=lag<=12):
                    candidates.append((b,lag))
            if candidates:
                b,lag=sorted(candidates,key=lambda z:z[0]["start"])[0]
                rows.append({**ids,"SGI3_event_id":a["event_id"],"SGI3_start":a["start"],"SGI3_end":a["end"],
                             "propagated":1,"SGI12_event_id":b["event_id"],"SGI12_start":b["start"],
                             "SGI12_end":b["end"],"onset_lag_months":lag})
            else:
                rows.append({**ids,"SGI3_event_id":a["event_id"],"SGI3_start":a["start"],"SGI3_end":a["end"],
                             "propagated":0,"SGI12_event_id":np.nan,"SGI12_start":pd.NaT,
                             "SGI12_end":pd.NaT,"onset_lag_months":np.nan})
    return pd.DataFrame(rows)

In [ ]:
well_event_rows=[]
for (pid,cluster,window),df in sgi.groupby(["piezometer_id","cluster_id","window_months"],sort=True):
    well_event_rows.extend(detect_events(df,"SGI_normal_score",
                                         {"piezometer_id":pid,"cluster_id":cluster,"window_months":window}))
well_events=pd.DataFrame(well_event_rows)

well_metric_rows=[]
for (pid,cluster,window),df in sgi.groupby(["piezometer_id","cluster_id","window_months"],sort=True):
    ev=well_events[(well_events["piezometer_id"]==pid)&
                   (well_events["cluster_id"]==cluster)&
                   (well_events["window_months"]==window)]
    well_metric_rows.append(compute_metrics(df,ev,"SGI_normal_score",
                                            {"piezometer_id":pid,"cluster_id":cluster,"window_months":window}))
well_metrics=pd.DataFrame(well_metric_rows)

well_matches=propagation_matches(well_events,["piezometer_id","cluster_id"])
well_probability=(well_matches.groupby(["piezometer_id","cluster_id"])["propagated"]
                  .mean().rename("propagation_probability_3_to_12").reset_index())
well_metrics=well_metrics.merge(well_probability,on=["piezometer_id","cluster_id"],how="left")
well_metrics.loc[well_metrics["window_months"]!=3,"propagation_probability_3_to_12"]=np.nan

In [ ]:
valid_group=group_sgi.dropna(subset=["group_mean_SGI"]).copy()

group_event_rows=[]
for (group,window),df in valid_group.groupby(["group","window_months"],sort=True):
    group_event_rows.extend(detect_events(df,"group_mean_SGI",
                                          {"group":group,"window_months":window}))
group_events=pd.DataFrame(group_event_rows)

sizes=metadata.groupby("cluster_id")["piezometer_id"].nunique().to_dict()
group_metric_rows=[]
for (group,window),df in valid_group.groupby(["group","window_months"],sort=True):
    ev=group_events[(group_events["group"]==group)&(group_events["window_months"]==window)]
    rec=compute_metrics(df,ev,"group_mean_SGI",{"group":group,"window_months":window})
    n=int(sizes[int(group)] if int(group) in sizes else sizes[str(group)])
    rec["group_wells"]=n
    rec["minimum_support"]=max(2,math.ceil(0.5*n))
    group_metric_rows.append(rec)

group_metrics=pd.DataFrame(group_metric_rows)
group_metrics=group_metrics[["group","window_months","group_wells","minimum_support","valid_months",
                             "drought_months","drought_fraction","N_events","mean_duration","max_duration",
                             "mean_severity_signed","mean_severity_deficit","mean_intensity","minimum_sgi",
                             "CRT","CRS","VUL","median_recovery_months","recovered_events","unrecovered_events"]]

group_matches=propagation_matches(group_events,["group"])
group_probability=(group_matches.groupby("group")["propagated"]
                   .mean().rename("propagation_probability_3_to_12").reset_index())
group_metrics=group_metrics.merge(group_probability,on="group",how="left")
group_metrics.loc[group_metrics["window_months"]!=3,"propagation_probability_3_to_12"]=np.nan

In [ ]:
outputs={
    "drought_events_well.csv":well_events,
    "drought_metrics_well.csv":well_metrics,
    "drought_events_group.csv":group_events,
    "drought_metrics_group.csv":group_metrics,
    "propagation_matches_well.csv":well_matches,
    "propagation_matches_group.csv":group_matches,
}
for name,df in outputs.items():
    path=PROCESSED/name
    df.to_csv(path,index=False)
    print(f"Saved {len(df):,} rows: {path}")

In [ ]:
display(group_metrics)
print("\nWell-level events:",len(well_events))
print("Group-level events:",len(group_events))